# Test normality of data before the analysis

Two most common tests include:

* Shapiro-Wilk test
* Anderson-Darling test

## Imports

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats

## Shapiro-Wilk test

* It is used to check the assumption of normality of the data
  * It can only detect normality. To test other distributions, check Anderson-Darling test
* It only works with continous data

The question:

> Is my data approximately normal?

The null hypothesis:

> H0: the data is distributed normally.

The alternative hypothesis:

> HA: the data is not distributed normally.

### Data

In [2]:
MEAN = 0
STDEV = 1
SAMPLE_SIZE = 50

# np.random.seed(0)
normal_data = np.random.normal(
    loc=MEAN,
    scale=STDEV,
    size=SAMPLE_SIZE
)

### Test

Documentation: [link](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.shapiro.html)

* `W_stat`: test statistics for Shapiro-Wilk test ranging from `0` to `1`
  * `W_stat = 1`: data fits normal distribution
  * `W_stat = 0`: data deviates from normal distribution significantly

* `p_val`: p-value

In [3]:
def test_shapiro_wilk(data, alpha = 0.05):
    """Calculate statistics for Shapiro-Wilk test for normality"""
    # Get sample size
    n = len(data)

    # Get test statistics
    W_stat, p_val = stats.shapiro(data)

    if p_val < alpha:
        reject_null_h = True
    else:
        reject_null_h = False

    return n, W_stat, p_val, alpha, reject_null_h

In [4]:
def report_shapiro_wilk(data, alpha = 0.05):
    """Produce a report for Shapiro-Wilk test for normality of data."""
    n, W_stat, p_val, alpha, _ = test_shapiro_wilk(data)

    results = {
        "sample size n": n,
        "W-statistic": f"{W_stat:.4f}",
        "p-value": f"{p_val:.4f}",
        "alpha": f"{alpha:.4f}",
    }

    # Apply uniform text alignment to all items in the report
    for label, value in results.items():
        print(f"{label:<15}: {value}")

    if p_val < alpha:
        print(f"{'Decision':<15}: p-value < α ({p_val:.4f} < {alpha:.4f})")
        print(f"{'Conclusion':<15}: Reject H₀. The data show evidence of non-normality.")
    else:
        print(f"{'Decision':<15}: p-value ≥ α ({p_val:.4f} ≥ {alpha:.4f})")
        print(f"{'Conclusion':<15}: Fail to reject H₀. The data are consistent with a normal distribution.")

In [5]:
report_shapiro_wilk(normal_data);


sample size n  : 50
W-statistic    : 0.9867
p-value        : 0.8429
alpha          : 0.0500
Decision       : p-value ≥ α (0.8429 ≥ 0.0500)
Conclusion     : Fail to reject H₀. The data are consistent with a normal distribution.


### Sensitivity check

The Shapiro-Wilk test works generally well. But as we get more data points, it can become over sensitive to the small deviations in the tail ends of the distribution.

Let's re-run the same experiment with a sample size of 5000 to check how often the null hypothesis is rejected (despite the fact the data was generated using normal distribution).

In [6]:
MEAN = 0
STDEV = 1
SAMPLE_SIZE = 5000  # Big sample size

samples = []

# Rerun experiment 1000 times
for i in range(1000):
    data = np.random.normal(
        loc=MEAN,
        scale=STDEV,
        size=SAMPLE_SIZE,
    )
    n, W_stat, p_val, alpha, reject_null_h = test_shapiro_wilk(data)
    samples.append(
        [i, n, W_stat, p_val, alpha, reject_null_h]
    )

df_samples = pd.DataFrame(
    samples,
    columns=[
        "iteration",
        "n",
        "W_stat",
        "p_value",
        "alpha",
        "reject_null",
    ],
)

In [7]:
df_samples

,iteration,n,W_stat,p_value,alpha,reject_null
0,0,5000,0.999831,0.979501,0.05,False
1,1,5000,0.999509,0.227708,0.05,False
2,2,5000,0.999378,0.085649,0.05,False
3,3,5000,0.999710,0.735149,0.05,False
4,4,5000,0.999723,0.772859,0.05,False
...,...,...,...,...,...,...
995,995,5000,0.999679,0.642725,0.05,False
996,996,5000,0.999420,0.117620,0.05,False
997,997,5000,0.999728,0.788811,0.05,False
998,998,5000,0.999675,0.631503,0.05,False


In [8]:
df_samples["reject_null"].value_counts()

reject_null
False    965
True      35
Name: count, dtype: int64

## Anderson-Darling test

### Data

In [9]:
MEAN = 0
STDEV = 1
SAMPLE_SIZE = 50

# np.random.seed(0)
normal_data = np.random.normal(
    loc=MEAN,
    scale=STDEV,
    size=SAMPLE_SIZE
)

### Test

Documentation: [link](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.anderson.html)

* `A^2` (*A square*): test statistics for Anderson-Darling test
  * `low A^2`: data fits the target distribution
  * `high A^2`: data deviates from the target distribution

In [10]:
A_square_stat, p_val = stats.anderson(
    x=data,
    dist="norm",
    method="interpolate"
)

In [11]:
A_square_stat

np.float64(0.22169086745998356)

In [12]:
p_val

np.float64(0.15)